# ProteinMind: Master Notebook for Google Colab

Multi-agent arginine/lysine methylation prediction platform built on **HybridARGME** (custom deep transformer with BMAM + AASSPE).

### Pipeline Overview:
1. **Mount Drive & install packages**
2. **Clone/upload project code and dataset**
3. **Train HybridARGME ensemble** (5-fold CV + SWA)
4. **Train ConsensusAgent meta-learner** (automatic after Step 3)
5. **Run multi-agent agentic analysis**
6. **Evaluate: Baseline vs Agentic**
7. **View visualizations**
8. **Backup to Drive**

## Step 1: Mount Google Drive & Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Install all dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers reportlab fastapi uvicorn matplotlib seaborn pandas numpy scikit-learn requests biopython python-multipart scipy joblib tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 124.7 MB/s eta 0:00:00


## Step 2: Set Up Project and Dataset Paths

In [ ]:
import os, sys

# ── CONFIGURE THESE TO MATCH YOUR DRIVE ──────────────────────────────────────
# Where your ProteinMind_Agent folder is in Google Drive
DRIVE_PROJECT_PATH  = "/content/drive/MyDrive/ProteinMind_Agent"

# Where your data-11 folder is in Google Drive
DRIVE_DATA_PATH     = "/content/drive/MyDrive/ProteinMind_Agent/data-11"
# ──────────────────────────────────────────────────────────────────────────────

# Symlink project code to /content for fast I/O
if os.path.exists(DRIVE_PROJECT_PATH) and not os.path.exists("/content/drive/MyDrive/ProteinMind_Agent"):
    os.symlink(DRIVE_PROJECT_PATH, "/content/drive/MyDrive/ProteinMind_Agent")
    print("[INFO] Symlinked project to /content/drive/MyDrive/ProteinMind_Agent")

# Symlink data-11 to /content/data-11 for fast training I/O
if os.path.exists(DRIVE_DATA_PATH) and not os.path.exists("/content/drive/MyDrive/ProteinMind_Agent/data-11"):
    os.symlink(DRIVE_DATA_PATH, "/content/drive/MyDrive/ProteinMind_Agent/data-11")
    print("[INFO] Symlinked data-11 to /content/drive/MyDrive/ProteinMind_Agent/data-11")
elif not os.path.exists(DRIVE_DATA_PATH):
    print(f"[WARNING] data-11 not found at {DRIVE_DATA_PATH}. Please upload your dataset.")

# Add project to Python path
PROJECT_DIR = "/content/drive/MyDrive/ProteinMind_Agent" if os.path.exists("/content/drive/MyDrive/ProteinMind_Agent") else DRIVE_PROJECT_PATH
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)
print(f"[INFO] Working directory: {os.getcwd()}")
print(f"[INFO] data-11 directory: {'/content/drive/MyDrive/ProteinMind_Agent/data-11' if os.path.exists('/content/drive/MyDrive/ProteinMind_Agent/data-11') else DRIVE_DATA_PATH}")

[INFO] Working directory: /content/drive/MyDrive/ProteinMind_Agent
[INFO] data-11 directory: /content/drive/MyDrive/ProteinMind_Agent/data-11


## Step 3: Train HybridARGME Ensemble + ConsensusAgent Meta-Learner

This runs 5-fold cross-validation with SWA, saves `checkpoints/pretrained_esm_v4.pt`, then automatically trains the `ConsensusAgent` meta-learner and saves `checkpoints/consensus_agent_meta_learner.pkl`.

**~15–30 minutes on Colab T4 GPU.**

In [ ]:
!python train.py

Streaming output truncated to the last 5000 lines.
[HybridARGME Agent] Scoring 2 candidate sites...
[HybridARGME Agent] Scoring 4 candidate sites...
[HybridARGME Agent] Scoring 1 candidate sites...
[HybridARGME Agent] Scoring 2 candidate sites...
[HybridARGME Agent] Scoring 6 candidate sites...
[HybridARGME Agent] Scoring 1 candidate sites...
[HybridARGME Agent] Scoring 2 candidate sites...
[HybridARGME Agent] Scoring 3 candidate sites...
[HybridARGME Agent] Scoring 1 candidate sites...
[HybridARGME Agent] Scoring 2 candidate sites...
[HybridARGME Agent] Scoring 3 candidate sites...
[HybridARGME Agent] Scoring 2 candidate sites...
[HybridARGME Agent] Scoring 3 candidate sites...
[HybridARGME Agent] Scoring 3 candidate sites...
[HybridARGME Agent] Scoring 2 candidate sites...
[HybridARGME Agent] Scoring 3 candidate sites...
[HybridARGME Agent] Scoring 1 candidate sites...
[HybridARGME Agent] Scoring 3 candidate sites...
[HybridARGME Agent] Scoring 4 candidate sites...
[HybridARGME Agent

## (Optional) Step 3b: Train ConsensusAgent Separately

If you already have a trained checkpoint and just want to (re)train the meta-learner:

In [ ]:
%cd /content/drive/MyDrive/ProteinMind_Agent

/content


In [ ]:
!python agents/train_consensus.py \
    --checkpoint checkpoints/pretrained_esm_v4.pt \
    --data-dir "/content/drive/MyDrive/ProteinMind_Agent/data-11"

Streaming output truncated to the last 5000 lines.
[HybridARGME Agent] Scoring 2 candidate sites...
[HybridARGME Agent] Scoring 1 candidate sites...
[HybridARGME Agent] Scoring 1 candidate sites...
[HybridARGME Agent] Scoring 2 candidate sites...
[HybridARGME Agent] Scoring 4 candidate sites...
[HybridARGME Agent] Scoring 1 candidate sites...
[HybridARGME Agent] Scoring 2 candidate sites...
[HybridARGME Agent] Scoring 6 candidate sites...
[HybridARGME Agent] Scoring 1 candidate sites...
[HybridARGME Agent] Scoring 2 candidate sites...
[HybridARGME Agent] Scoring 3 candidate sites...
[HybridARGME Agent] Scoring 1 candidate sites...
[HybridARGME Agent] Scoring 2 candidate sites...
[HybridARGME Agent] Scoring 3 candidate sites...
[HybridARGME Agent] Scoring 2 candidate sites...
[HybridARGME Agent] Scoring 3 candidate sites...
[HybridARGME Agent] Scoring 3 candidate sites...
[HybridARGME Agent] Scoring 2 candidate sites...
[HybridARGME Agent] Scoring 3 candidate sites...
[HybridARGME Agent

In [ ]:
!ls data-11

Chimpanzee  dataset.tsv  full_arginine_data.tsv  Human	Mouse  Rat


## Step 4: Run Agentic Analysis (CLI Query)

In [ ]:
!python agents/create_placeholder_metadata.py

Created: /content/drive/MyDrive/ProteinMind_Agent/data-11/Human/Asymmetric_dimethylarginine/human_positive_Asymmetric_dimethylarginine_metadata.csv
Created: /content/drive/MyDrive/ProteinMind_Agent/data-11/Human/Asymmetric_dimethylarginine/human_negative_Asymmetric_dimethylarginine_metadata.csv
Created: /content/drive/MyDrive/ProteinMind_Agent/data-11/Human/Dimethylated_arginine/human_positive_Dimethylated_arginine_metadata.csv
Created: /content/drive/MyDrive/ProteinMind_Agent/data-11/Human/Dimethylated_arginine/human_negative_Dimethylated_arginine_metadata.csv
Created: /content/drive/MyDrive/ProteinMind_Agent/data-11/Human/Omega-N-methylarginine/human_positive_Omega-N-methylarginine_metadata.csv
Created: /content/drive/MyDrive/ProteinMind_Agent/data-11/Human/Omega-N-methylarginine/human_negative_Omega-N-methylarginine_metadata.csv
Created: /content/drive/MyDrive/ProteinMind_Agent/data-11/Human/Symmetric_dimethylarginine/human_positive_Symmetric_dimethylarginine_metadata.csv
Created: /

In [ ]:
!python run_proteinmind.py \
    --query "Analyze sequence: MKRGGKPLRTARGGRVVTSMKRGGKPLRTA" \
    --checkpoint checkpoints/pretrained_esm_v4.pt \
    --metadata-dir "/content/drive/MyDrive/ProteinMind_Agent/data-11" \
    --species Human \
    --ptm Asymmetric_dimethylarginine \
    --label positive \
    --output-dir proteinmind_outputs

Traceback (most recent call last):
  File "/content/drive/MyDrive/ProteinMind_Agent/run_proteinmind.py", line 7, in <module>
    from visuals.generator import VisualsGenerator
  File "/content/drive/MyDrive/ProteinMind_Agent/visuals/generator.py", line 271
    print("[Visuals] Both APIs failed — generating synthetic helix fallback.")
IndentationError: expected an indented block after 'else' statement on line 270


In [ ]:
!python run_proteinmind.py \
    --query "Analyze sequence: MKRGGKPLRTARGGRVVTSMKRGGKPLRTA" \
    --checkpoint checkpoints/pretrained_esm_v4.pt \
    --metadata-dir "/content/drive/MyDrive/ProteinMind_Agent/data-11" \
    --species "Human" \
    --ptm "Asymmetric_dimethylarginine" \
    --label "positive"

Traceback (most recent call last):
  File "/content/drive/MyDrive/ProteinMind_Agent/run_proteinmind.py", line 7, in <module>
    from visuals.generator import VisualsGenerator
  File "/content/drive/MyDrive/ProteinMind_Agent/visuals/generator.py", line 271
    print("[Visuals] Both APIs failed — generating synthetic helix fallback.")
IndentationError: expected an indented block after 'else' statement on line 270


In [ ]:
!python agents/generate_clinical_report.py \
    --data-dir "/content/drive/MyDrive/ProteinMind_Agent/data-11" \
    --species "Human" \
    --ptm "Asymmetric_dimethylarginine" \
    --label "negative" \
    --output "human_negative_asymmetric_report.pdf"

python3: can't open file '/content/drive/MyDrive/ProteinMind_Agent/agents/generate_clinical_report.py': [Errno 2] No such file or directory


In [ ]:
!cat /content/drive/MyDrive/ProteinMind_Agent/data-11/Human/Asymmetric_dimethylarginine/human_positive_asymmetric_dimethylarginine_metadata.csv

cat: /content/drive/MyDrive/ProteinMind_Agent/data-11/Human/Asymmetric_dimethylarginine/human_positive_asymmetric_dimethylarginine_metadata.csv: No such file or directory


## Step 5: Evaluate Baseline vs Agentic Framework

In [ ]:
# Full comparative evaluation: HybridARGME standalone vs Multi-Agent Consensus
!python evaluate_framework.py \
    --checkpoint checkpoints/pretrained_esm_v4.pt \
    --dataset splits/combined_test.csv \
    --data-dir //content/drive/MyDrive/ProteinMind_Agent/data-11 \
    --output-dir proteinmind_outputs

  Evaluating Standalone HybridARGME vs Multi-Agent Framework
[ERROR] Test dataset not found at: splits/combined_test.csv
Please run train.py first to generate the dataset splits.


In [ ]:
# Thesis-style comparison table (sample-based)
!python evaluate_thesis.py \
    --checkpoint checkpoints/pretrained_esm_v4.pt \
    --data-dir /content/data-11 \
    --num-samples 200

🎓 THESIS EVALUATION: Baseline vs Agentic Framework

[1] Loading Test Data from: /content/drive/MyDrive/ProteinMind_Agent/data-11/dataset.tsv
❌ Error: Could not find 'sequence' and 'label' columns. Found: ['Entry', 'Organism', 'sequence', 'Modified residue']


## Step 6: View Visualizations Inline

In [ ]:
from IPython.display import Image, display
import os

output_dir = "proteinmind_outputs"
if os.path.exists(output_dir):
    png_files = sorted(f for f in os.listdir(output_dir) if f.endswith(".png"))
    if png_files:
        for file_name in png_files:
            print(f"\n=== {file_name} ===")
            display(Image(os.path.join(output_dir, file_name)))
    else:
        print("No .png files found. Did the analysis run successfully?")
else:
    print("[ERROR] No output folder found.")

## Step 7: Render 3D Protein Structure Viewer

In [ ]:
import IPython
import glob, os

html_files = (
    glob.glob("proteinmind_outputs/*.html") or
    glob.glob("proteinmind_outputs/**/*.html", recursive=True)
)
if html_files:
    viewer_path = html_files[0]
    print(f"Loading: {viewer_path}")
    display(IPython.display.HTML(filename=viewer_path))
else:
    print("[ERROR] No 3D structure viewer file found.")

Loading: proteinmind_outputs/structure_viewer_P04637.html


## Step 8: Backup Checkpoints and Reports to Drive

In [ ]:
BACKUP_DIR = "/content/drive/MyDrive/proteinmind_backups"
os.makedirs(BACKUP_DIR, exist_ok=True)
!cp -r checkpoints/* "$BACKUP_DIR/" 2>/dev/null && echo 'Checkpoints backed up ✓'
!cp -r proteinmind_outputs/* "$BACKUP_DIR/" 2>/dev/null && echo 'Outputs backed up ✓'
print(f"[INFO] Backup complete → {BACKUP_DIR}")